# 01 - Build the analysis table

**Goal:** one tidy GeoDataFrame, one row per block group, saved to
`data/processed/analysis_table.gpkg`. Everything downstream reads only this.

**What you'll learn:** projections, spatial joins, areal interpolation, and the
habit of collapsing messy inputs into a single clean table before you model.

---

## 1. Reproject everything

Geographic coordinates (EPSG:4326, lat/lon) are **degrees**, not metres. Areas
and distances computed in them are meaningless. Reproject to a projected CRS
appropriate to Denver before any measurement:

```python
CRS = 26953  # NAD83 / Colorado Central, metres
gdf = gdf.to_crs(CRS)
```

Confirm this is the right zone for Denver before you rely on it; the city
publishes its own preferred CRS, and matching it avoids arguments later.


In [ ]:
import geopandas as gpd
from canopy.io import RAW, PROCESSED

CRS = 26953
# TODO: load raw layers, reproject all to CRS


## 2. Compute canopy fraction per block group

The operation you want is an **overlay** (intersection) of canopy polygons with
block-group polygons, then a sum of intersected areas grouped by block group:

```python
inter = gpd.overlay(canopy, bg[["GEOID", "geometry"]], how="intersection")
inter["canopy_m2"] = inter.area
gained = inter.groupby("GEOID")["canopy_m2"].sum()
```

Watch for: invalid geometries (`.make_valid()`), canopy polygons that straddle
block-group borders, and block groups with zero canopy that vanish from the
groupby and must be filled back in with 0.


In [ ]:
# TODO: overlay, aggregate, compute canopy_frac = canopy_m2 / bg_area_m2


## 3. Attach demographics

Join the ACS table on `GEOID`. Check the join actually matched - a silent
partial join is the single most common way a spatial analysis goes quietly
wrong. Assert that the number of matched rows equals the number of block groups.


In [ ]:
# TODO: merge ACS; assert no unmatched GEOIDs


## 4. Estimate planting capacity

This is the parameter that will decide your whole answer, so derive it, don't
guess it. Options, roughly in order of effort:

1. **Crude:** capacity proportional to street-centreline length in the block
   group (one tree per 9 m of frontage is a common planning rule of thumb).
2. **Better:** subtract existing canopy, buildings, and pavement from total area
   to get plantable area; divide by a mature crown footprint.
3. **Best:** use parcel + right-of-way data to separate public plantable space
   from private.

Whichever you pick, write down the assumption explicitly in a markdown cell.
Reviewers care far more about a stated assumption than a hidden sophisticated one.


In [ ]:
# TODO: compute capacity_k per block group


## 5. Calibrate crown area

You need `a` = canopy area added per mature tree, in m². Rather than pulling a
number from a paper, calibrate it from Denver's own data: the tree inventory
gives you counts and diameters, the canopy layer gives you total area. Compare
what a diameter-based allometric estimate predicts against observed canopy in
well-inventoried areas.

Note honestly where this breaks down: the inventory is public trees only, so the
ratio is contaminated by private canopy.


In [ ]:
# TODO: estimate crown area per tree; record the value and its uncertainty


In [ ]:
# TODO: save the finished table
# table.to_file(PROCESSED / "analysis_table.gpkg", driver="GPKG")
